# Errors Claude Code Noticed — 3/4 8:11pm

Code review of `index.html` (10,477 lines, ~563KB of JS).

**Syntax:** Clean. Node.js `--check` passes with 0 errors.

**Bugs found:** 3 confirmed logic bugs + 1 dead constant.

---

## BUG 1 (Real Impact) — Factory Level Punch Kill: Missing `insulin` + `glitch` Type Rewards

**Location:** `createLevelScene` function → punch kill handler, ~line 3392

**What happens:** When a player punch-kills an `insulin` or `glitch` enemy in any factory-created level (Worlds 4, 5, and 6), the kill reward falls through to the default `ALOE.mochiDeath` (20 aloe) instead of the correct values.

- `ALOE.insulinDeath` = **55**
- `ALOE.glitchDeath` = **70**
- `ALOE.mochiDeath` = **20** ← what they actually get

**Worlds affected:** 4, 5, 6 — all use `createLevelScene` and all have `insulin`/`glitch` enemies.

**The broken chain (line 3392):**
```js
// HAS: rasta, ghost, bomber, tank, daikon, thrower
// MISSING: insulin, glitch  <-- pays mochiDeath (20) instead of 55/70
m.type==='rasta' ? ALOE.rastaDeath :
m.type==='ghost' ? ALOE.ghostDeath :
m.type==='bomber' ? ALOE.bomberDeath :
m.type==='tank' ? ALOE.tankDeath :
m.type==='daikon' ? ALOE.daikonDeath :
m.type==='thrower' ? ALOE.throwerDeath :
ALOE.mochiDeath   // <-- insulin and glitch fall here
```

**Fix:** Add `m.type==='insulin' ? ALOE.insulinDeath : m.type==='glitch' ? ALOE.glitchDeath :` before the final `ALOE.mochiDeath` fallback.

## BUG 2 (Real Impact) — Factory Level Kick Kill: Same Missing Types

**Location:** `createLevelScene` → kick kill handler, ~line 3407

**Same issue as Bug 1 but for kicks.** The kick kill reward chain also lacks `insulin` and `glitch` type checks.

Players kicking `insulin` or `glitch` enemies in Worlds 4-6 also get 20 aloe instead of 55/70.

Note: `km.type==='glitch'&&km._hittable===false` guard exists (line ~3399) to skip unhittable glitch phases — so glitch CAN be kicked when hittable, but the kill reward is wrong.

## BUG 3 (Minor) — Level11 Kick Kill: Missing `insulin` + `glitch` Type Rewards

**Location:** `Level11Scene` kick kill handler, ~line 2105

**The Level11 PUNCH kill (line 2087) correctly handles insulin and glitch. The Level11 KICK kill (line 2105) does not.**

```js
// Level11 PUNCH kill (correct):
// rasta, ghost, bomber, tank, daikon, insulin, glitch -> mochiDeath fallback

// Level11 KICK kill (wrong):
// rasta, ghost, bomber, tank, daikon -> mochiDeath fallback  (insulin/glitch MISSING)
```

**Practical impact:** Level11 itself doesn't spawn insulin or glitch enemies, so this is currently dormant. But it's an inconsistency that could bite if enemies are ever added to that level.

## DEAD CODE — `ALOE.rastaEliteDeath` Defined, Never Used

**Location:** `ALOE` constants object, line 68

```js
rastaEliteDeath: 80,  // defined here
```

**Zero references** to `ALOE.rastaEliteDeath` anywhere in the 10k-line codebase. No enemy of `type: 'rastaElite'` exists either.

This looks like a planned elite Rasta Corp enemy that was never implemented, or was renamed/removed at some point. Either wire it up or remove it to avoid confusion.

## Summary Table

| # | Type | Location | Impact | Fix Complexity |
|---|------|----------|--------|----------------|
| 1 | Logic bug | `createLevelScene` punch kill ~L3392 | Players under-rewarded (20 vs 55-70 aloe) on insulin/glitch kills, Worlds 4-6 | Easy — add 2 ternary checks |
| 2 | Logic bug | `createLevelScene` kick kill ~L3407 | Same as above for kick attacks | Easy — same fix |
| 3 | Logic bug | `Level11Scene` kick kill ~L2105 | Dormant — L11 has no insulin/glitch enemies currently | Easy — same fix pattern |
| 4 | Dead code | `ALOE.rastaEliteDeath` L68 | Unused constant — no `rastaElite` enemy type exists | Remove or implement |

---

**No syntax errors. No broken scene references. All 35+ scenes are properly registered in the Phaser game config.**

*— Claude Code, 3/4 8:11pm*